# Week 6: SVD, gradients, and gradient descent

**Budget:** about 10 hours.

| Session | Topic | Time |
|---|---|---|
| Day 1 | SVD: rotate → stretch → rotate; low-rank approximation | 2.5 h |
| Day 2 | *(optional)* Latent semantic analysis: SVD finds topics in text | 2 h |
| Day 3 | Derivatives, the chain rule, gradients, gradient checking | 2.5 h |
| Day 4 | Gradient descent: learning rates, divergence, curvature | 2.5 h |
| Review | Spaced recall and a blank-page rebuild | 0.5 h |

**Watch:** 3Blue1Brown, *Essence of Calculus*, chapters 1–4 (before Day 3). **Read:** *Mathematics for Machine Learning*, sections 4.5 (SVD) and 5.1–5.2 (calculus).

### Done when
- [ ] Every test cell prints ✅.
- [ ] You can explain the SVD as three geometric steps, and say why truncating it gives the best low-rank approximation (intuition is enough).
- [ ] You can derive the gradient of $\|Xw - y\|^2$ on paper.
- [ ] You can look at a loss curve and say "learning rate too high", "too low", or "fine".

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

---
# Day 1: Singular value decomposition (2.5 h)

Eigenvectors only exist for square matrices, and they're only guaranteed to be nice for symmetric ones. The **SVD** works for **any** matrix $A$ ($m \times n$):
$$A = U \Sigma V^\top$$
Geometrically, **every** linear map is: **rotate** ($V^\top$) → **stretch along the axes** ($\Sigma$, the singular values $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$) → **rotate** ($U$).

**Link to Week 5:** the columns of $V$ are the eigenvectors of $A^\top A$, and $\sigma_i^2$ are its eigenvalues. So PCA can be computed with the SVD of the centred data, which is how scikit-learn does it.

**Why it matters:** keeping only the largest $k$ singular values gives $A_k$, the **best rank-$k$ approximation** of $A$ (the Eckart–Young theorem). This is compression, denoising, recommender systems, latent semantic analysis, and the idea behind LoRA fine-tuning of large language models.

In [ ]:
A = np.array([[3.0, 1.0], [1.0, 3.0], [0.0, 2.0]])
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print("U\n", U, "\nsingular values", s, "\nVt\n", Vt)
print("reconstruction error:", np.abs(U @ np.diag(s) @ Vt - A).max())
print("σ² vs eigenvalues of AᵀA:", s ** 2, np.linalg.eigvalsh(A.T @ A)[::-1])

### Exercise 6.1: `low_rank`
Return the rank-`k` approximation $A_k = U_k \Sigma_k V_k^\top$. Then check the Eckart–Young error formula: $\|A - A_k\|_F = \sqrt{\sum_{i>k} \sigma_i^2}$.

In [ ]:
def low_rank(A, k):
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
M = rng.normal(size=(40, 25))
s = np.linalg.svd(M, compute_uv=False)
for k in [1, 5, 25]:
    err = np.linalg.norm(M - low_rank(M, k))
    assert abs(err - np.sqrt((s[k:] ** 2).sum())) < 1e-8
assert np.linalg.matrix_rank(low_rank(M, 3)) == 3
print("✅ 6.1 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

`U, s, Vt = np.linalg.svd(A, full_matrices=False)`, then `U[:, :k] @ np.diag(s[:k]) @ Vt[:k]`.

</details>


### Image compression
An image is just a matrix. How many singular values do you need before it looks right?

In [ ]:
from sklearn.datasets import load_sample_image
img = load_sample_image("china.jpg").mean(axis=2)          # greyscale, shape (427, 640)
s_img = np.linalg.svd(img, compute_uv=False)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, k in zip(axes, [5, 20, 50, 427]):
    ax.imshow(low_rank(img, k), cmap="gray"); ax.axis("off")
    stored = k * (img.shape[0] + img.shape[1] + 1) / img.size
    ax.set_title(f"k={k}, {stored:.0%} of the numbers")
plt.show()
plt.semilogy(s_img); plt.title("singular values (log scale)"); plt.xlabel("i"); plt.show()

✍️ *Why do the singular values drop so quickly for a natural image? What would the curve look like for pure random noise?* (double-click to edit)



---
# Day 2 *(optional)*: Latent semantic analysis (2 h)

Build a **term–document matrix** $M$ (rows are words, columns are documents). Its SVD groups words that occur in *similar documents* into shared "topic" directions. In the reduced space, a query for "tumor" can match a document about "cancer" even when the word "tumor" never appears in it. That's the first step from *lexical* search (Week 10) to *semantic* search (Week 11).

In [ ]:
docs = [
    "tumor growth in breast cancer",
    "cancer chemotherapy and tumor response",
    "oncology cancer treatment outcomes",
    "insulin and glucose in diabetes",
    "blood glucose control with insulin therapy",
    "diabetes diet and blood sugar",
]

### Exercise 6.2: `term_doc_matrix`
Return `(M, vocab)`, where `vocab` is the sorted list of unique words and `M[i, j]` counts word `vocab[i]` in document `j`. Split on whitespace.

In [ ]:
def term_doc_matrix(docs):
    raise NotImplementedError

In [ ]:
M, vocab = term_doc_matrix(docs)
assert M.shape == (len(vocab), 6) and vocab == sorted(vocab)
assert M[vocab.index("cancer"), 1] == 1 and M[vocab.index("glucose"), 4] == 1
U, s, Vt = np.linalg.svd(M, full_matrices=False)
doc_vecs = (np.diag(s[:2]) @ Vt[:2]).T          # each document in the 2-D "topic" space
plt.scatter(doc_vecs[:, 0], doc_vecs[:, 1])
for i, d in enumerate(docs): plt.annotate(d[:22], doc_vecs[i])
plt.title("documents in LSA space"); plt.show()

q = np.zeros(len(vocab)); q[vocab.index("oncology")] = 1          # query: "oncology"
q_vec = U[:, :2].T @ q
sims = doc_vecs @ q_vec / (np.linalg.norm(doc_vecs, axis=1) * np.linalg.norm(q_vec))
print("similarity of each doc to 'oncology':", sims.round(2))
print("✅ 6.2 passed")

✍️ *Document 0 never contains 'oncology'. Why does it still score highly?* (double-click to edit)



---
# Day 3: Derivatives, the chain rule, and gradients (2.5 h)

- The **derivative** $f'(x)$ is the slope: how much $f$ changes per unit change in $x$, *locally*.
- For a function of many inputs $f(w_1, \dots, w_d)$, the **gradient** $\nabla f = (\partial f / \partial w_1, \dots, \partial f / \partial w_d)$ is a vector pointing in the direction of **steepest increase**. To minimise a loss, step in the opposite direction.
- **Chain rule**: if $L = f(g(w))$, then $\frac{dL}{dw} = f'(g(w)) \cdot g'(w)$. Backpropagation (Week 13) is nothing more than the chain rule applied carefully.

### Worked derivation: the gradient of the squared error
$L(w) = \|Xw - y\|^2 = \sum_i (x_i^\top w - y_i)^2$. Let $r = Xw - y$ (the residuals). By the chain rule, $\frac{\partial L}{\partial w_j} = \sum_i 2 r_i X_{ij}$, so
$$\nabla L = 2 X^\top (Xw - y).$$
Setting it to zero gives $X^\top X w = X^\top y$, the normal equations from Week 5. **Do this derivation on paper now**, then check it numerically.

### Exercise 6.3: `numerical_grad`
Estimate the gradient of a scalar function `f` at the vector `w` using central differences, one coordinate at a time. This is your **gradient checker**: whenever you derive a gradient by hand, from now on, you test it against this.

In [ ]:
def numerical_grad(f, w, h=1e-6):
    raise NotImplementedError

In [ ]:
f = lambda w: w[0] ** 2 + 3 * w[0] * w[1] + np.sin(w[1])
w0 = np.array([1.0, 2.0])
analytic = np.array([2 * w0[0] + 3 * w0[1], 3 * w0[0] + np.cos(w0[1])])
assert np.allclose(numerical_grad(f, w0), analytic, atol=1e-6)
print("✅ 6.3 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Loop over `j in range(len(w))`. Make `e = np.zeros_like(w)` with `e[j] = h`.

</details>

<details><summary>Hint 2</summary>

`g[j] = (f(w + e) - f(w - e)) / (2 * h)`. Make sure `w` is a float array, or adding h gets lost.

</details>


### Exercise 6.4: `mse_loss` and `mse_grad`
Implement $L(w) = \frac{1}{n}\|Xw - y\|^2$ (note the $1/n$: the *mean* squared error) and its gradient. Check them with your gradient checker.

In [ ]:
def mse_loss(w, X, y):
    raise NotImplementedError

def mse_grad(w, X, y):
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(1)
X, y, w = rng.normal(size=(30, 4)), rng.normal(size=30), rng.normal(size=4)
num = numerical_grad(lambda v: mse_loss(v, X, y), w)
assert np.allclose(mse_grad(w, X, y), num, atol=1e-5)
print("✅ 6.4 passed: your analytic gradient matches the numerical one")

---
# Day 4: Gradient descent (2.5 h)

$$w_{t+1} = w_t - \eta \, \nabla L(w_t)$$
The **learning rate** $\eta$ is the most important hyperparameter in all of deep learning:
- too small → painfully slow,
- too large → overshoots and **diverges** (the loss explodes),
- "just right" depends on the **curvature**, which is given by the eigenvalues of the Hessian $\nabla^2 L$. For MSE that's $\frac{2}{n} X^\top X$, and gradient descent is stable only if $\eta < 2 / \lambda_{\max}$. That's Week 5 paying off.

### Exercise 6.5: `gradient_descent`
Run `n_steps` steps from `w0` and return the list of all iterates (including `w0`) as an array of shape (n_steps + 1, d).

In [ ]:
def gradient_descent(grad_fn, w0, lr, n_steps):
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(2)
X = np.c_[np.ones(200), rng.normal(size=(200, 2))]
y = X @ np.array([1.0, 2.0, -3.0]) + 0.1 * rng.normal(size=200)
hist = gradient_descent(lambda w: mse_grad(w, X, y), np.zeros(3), lr=0.1, n_steps=300)
assert hist.shape == (301, 3)
w_closed = np.linalg.solve(X.T @ X, X.T @ y)
assert np.allclose(hist[-1], w_closed, atol=1e-4)
print("✅ 6.5 passed: gradient descent reaches the closed-form solution", hist[-1])

### Experiment: three learning rates
First compute the stability limit $2/\lambda_{\max}$, then try learning rates below, near, and above it.

In [ ]:
H = 2 / len(X) * X.T @ X
lam_max = np.linalg.eigvalsh(H).max()
print(f"λ_max = {lam_max:.3f}  →  stable if lr < {2 / lam_max:.3f}")
for lr in [0.01, 0.9 * 2 / lam_max, 1.05 * 2 / lam_max]:
    h = gradient_descent(lambda w: mse_grad(w, X, y), np.zeros(3), lr=lr, n_steps=60)
    plt.semilogy([mse_loss(w, X, y) for w in h], label=f"lr={lr:.3f}")
plt.xlabel("step"); plt.ylabel("MSE (log scale)"); plt.legend(); plt.title("Learning rate vs convergence"); plt.show()

✍️ *Describe each curve. The middle one probably zig-zags before converging: why? What happens to the stable learning rate if you multiply one feature by 100 (and why does that make standardisation important)?* (double-click to edit)



---
# Review (30 min)

### Blank-page rebuild
Write `numerical_grad` and `gradient_descent` from memory, then minimise $f(w) = (w_0 - 3)^2 + 10(w_1 + 1)^2$ with them, using `numerical_grad` as the gradient.

In [ ]:
# your code here: define numerical_grad2 and gradient_descent2, then find w_star
w_star = None

In [ ]:
assert np.allclose(w_star, [3, -1], atol=1e-3)
print("✅ rebuild passed")

### Spaced recall
1. *(Week 3)* What's the big-O of binary search, and why?
2. *(Week 5)* What does an eigenvalue of 0.5 do to its eigenvector?
3. *(Week 5)* Write the three steps of PCA.
4. The SVD in three geometric words?
5. Why is $A_k$ from the truncated SVD useful? Name two applications.
6. What does the gradient point towards? Why do we subtract it?
7. What happens if the learning rate is above $2/\lambda_{\max}$?

✍️ *Your answer:* (double-click to edit)



### Commit, log, update the tracker